In [41]:
# %pip install delta-spark

In [42]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col

In [43]:
existing_spark = SparkSession.getActiveSession()
print(existing_spark)
if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

None


In [44]:
spark = (
        SparkSession.builder
        .appName("DeltaTable App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    );

In [45]:
spark

In [46]:
data = [
    (1, "Alice", 30, "2025-12-01"),
    (2, "Bob",   28, "2025-12-01"),
    (3, "Cleo",  35, "2025-12-02"),
]
cols = ["id", "name", "age", "dt"]

df = spark.createDataFrame(data, cols)

In [47]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|spark_catalog|
+-------------+



In [48]:
spark.sql("USE spark_catalog")

DataFrame[]

In [49]:
spark.sql("SELECT current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|    spark_catalog|
+-----------------+



In [52]:
spark.sql("CREATE DATABASE IF NOT EXISTS delta_tbl")

DataFrame[]

In [53]:
spark.sql("SHOW DATABASES").show()

+----------+
| namespace|
+----------+
|catalog_db|
|   default|
| delta_tbl|
|my_glue_db|
+----------+



In [54]:
spark.sql("USE DELTA_TBL")

26/04/28 14:18:23 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


DataFrame[]

In [55]:
df = df.withColumn("id", col("id").cast("long"))

In [56]:
df.show()

+---+-----+---+----------+
| id| name|age|        dt|
+---+-----+---+----------+
|  1|Alice| 30|2025-12-01|
|  2|  Bob| 28|2025-12-01|
|  3| Cleo| 35|2025-12-02|
+---+-----+---+----------+



In [57]:
spark.sql("DROP TABLE IF EXISTS DELTA_TBL.delta_exp_tbl")

DataFrame[]

In [58]:
df.write.format("delta").mode("overwrite").partitionBy("dt").saveAsTable("DELTA_TBL.delta_exp_tbl")

26/04/28 14:18:39 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`delta_tbl`.`delta_exp_tbl` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/04/28 14:18:39 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
26/04/28 14:18:39 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
26/04/28 14:18:39 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
26/04/28 14:18:39 WARN HiveConf: HiveConf of name hive.metastore.client.factory.class does not exist
26/04/28 14:18:39 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [59]:
spark.sql("SHOW TABLES").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|delta_tbl|delta_exp_tbl|      false|
|delta_tbl|    delta_src|      false|
+---------+-------------+-----------+



In [60]:
source_df = spark.createDataFrame([
    (1, "Alice", 30),
    (2, "Bob", 28),
    (4, "Dave", 35)
], ["id","name","age"])

In [61]:
spark.sql("DROP TABLE IF EXISTS DELTA_TBL.delta_src")

DataFrame[]

In [62]:
source_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("DELTA_TBL.delta_src")

26/04/28 14:18:44 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`delta_tbl`.`delta_src` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [63]:
if not DeltaTable.isDeltaTable(spark, spark.sql("DESCRIBE DETAIL DELTA_TBL.delta_src").collect()[0]['location']):
    target_initial = spark.createDataFrame([
        (1, "Alice", 29),   # age changed
        (2, "Bob", 28),
        (3, "Charlie", 40)
    ], ["id","name","age"])
    target_initial.write.format("delta").mode("overwrite").saveAsTable("DELTA_TBL.delta_src")

In [64]:
spark.sql("DESCRIBE DETAIL DELTA_TBL.delta_src").select("name", "location").show(truncate=False)

+---------------------------------+-------------------------------------------------------------+
|name                             |location                                                     |
+---------------------------------+-------------------------------------------------------------+
|spark_catalog.delta_tbl.delta_src|s3a://shivchoudhury-datasets/warehouse/delta_tbl.db/delta_src|
+---------------------------------+-------------------------------------------------------------+



In [65]:
delta_table = DeltaTable.forPath(spark, spark.sql("DESCRIBE DETAIL DELTA_TBL.delta_src").collect()[0]['location'])

In [66]:
delta_table.toDF().show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  4| Dave| 35|
|  2|  Bob| 28|
+---+-----+---+



In [67]:
source_df.show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  2|  Bob| 28|
|  4| Dave| 35|
+---+-----+---+



In [68]:
delta_table.alias("t").merge(source_df.alias("s"), "t.id = s.id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [69]:
spark.sql("select * from DELTA_TBL.delta_src").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  2|  Bob| 28|
|  4| Dave| 35|
+---+-----+---+



In [70]:
new_source_df = spark.createDataFrame([
    (1, "Alice", 50),
    (2, "Bob", 15),
    (5, "Rey", 25),
    (7, "Rashi", 30)
], ["id","name","age"])

In [71]:
delta_table.alias("t").merge(new_source_df.alias("s"), "t.id = s.id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [72]:
spark.sql("select * from DELTA_TBL.delta_src").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 50|
|  2|  Bob| 15|
|  4| Dave| 35|
|  5|  Rey| 25|
|  7|Rashi| 30|
+---+-----+---+



In [73]:
delta_table.delete("true") # deletes all data files but keeps metadata

In [74]:
spark.sql("select * from DELTA_TBL.delta_src").show()

+---+----+---+
| id|name|age|
+---+----+---+
+---+----+---+



In [77]:
spark.sql("DROP TABLE IF EXISTS DELTA_TBL.delta_src")
spark.sql("DROP TABLE IF EXISTS DELTA_TBL.delta_exp_tbl")

DataFrame[]

In [78]:
spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [79]:
spark.stop()